# ILO 2025 occupation exposure and labor-model scenario

This notebook follows the original analysis notebook’s interactive Plotly approach. It explores the ILO 2025 GenAI task-exposure scores after matching them to this project’s 464 occupation groups, compares those inputs with the project’s Frey–Osborne scores, and inspects the labor-model rerun.

The ILO score measures technical exposure of tasks to generative AI. It is not a probability that an occupation or job will disappear. Treating it as a demand-reduction fraction in the project’s model is a scenario assumption.

Run cells from top to bottom. Interactive charts support hover, zoom, pan, and legend toggling; standalone copies are written to the extension results folder.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "extensions" / "onet_task_exposure" / "results").exists():
    ROOT = ROOT.parent
RESULTS = ROOT / "extensions" / "onet_task_exposure" / "results"
INTERACTIVE = RESULTS / "interactive"
INTERACTIVE.mkdir(parents=True, exist_ok=True)
CONFIG = {"displaylogo": False, "scrollZoom": True, "responsive": True}

FIGURES = {}
def show_interactive(fig, filename):
    """Display and save a Plotly chart, retaining data for cached notebook output."""
    fig.write_html(INTERACTIVE / filename, include_plotlyjs=True, config=CONFIG)
    FIGURES[filename] = fig
    bundle = {
        "application/vnd.plotly.v1+json": json.loads(fig.to_json()),
        "text/html": fig.to_html(full_html=False, include_plotlyjs=True, config=CONFIG),
    }
    display(bundle, raw=True)
    print(f"Saved interactive chart: {INTERACTIVE / filename}")

scores = pd.read_csv(RESULTS / "ilo2025_project_occupation_scores.csv")
model_inputs = pd.read_csv(RESULTS / "ilo2025_model_inputs.csv")
outcomes = pd.read_csv(RESULTS / "ilo2025_vs_frey_osborne_simulation_by_occupation.csv")
ts_fo = pd.read_csv(RESULTS / "timeseries_frey_osborne.csv")
ts_ilo = pd.read_csv(RESULTS / "timeseries_ilo_2025.csv")

print(f"Project root: {ROOT}")
print(f"Occupations: {len(scores)}; crosswalked: {scores['ilo_2025_task_mean'].notna().sum()}; model-filled: {model_inputs['ilo_2025_model_estimate'].notna().sum()}")

## 1. Match coverage and ambiguity

The crosswalk runs from Census 2010 occupation codes through SOC 2010 to ISCO-08. Multiple candidate ISCO occupations are kept in the score file; the project score for those rows is their unweighted mean. The table shows how many occupations rely on one candidate versus several.

In [ ]:
coverage = (scores.groupby("match_status", as_index=False)
           .agg(occupations=("project_occ_id", "count"),
                mean_candidates=("candidate_count", "mean")))
display(coverage)

colors = {"unique_code_crosswalk": "#4776a8", "ambiguous_code_crosswalk_mean": "#e5a443", "unmatched": "#999999"}
fig = go.Figure()
for _, row in coverage.iterrows():
    fig.add_trace(go.Bar(
        x=[row["occupations"]], y=[row["match_status"].replace("_", " ")], orientation="h",
        name=row["match_status"].replace("_", " "),
        marker_color=colors.get(row["match_status"], "#777777"),
        customdata=[[row["mean_candidates"]]],
        hovertemplate="<b>%{y}</b><br>Project occupations: %{x}<br>Mean ISCO candidates: %{customdata[0]:.2f}<extra></extra>"))
fig.update_layout(title="ILO crosswalk coverage", template="plotly_white", barmode="stack",
                  xaxis_title="Project occupation groups", yaxis_title="Match status",
                  height=430, hovermode="closest", dragmode="zoom")
show_interactive(fig, "ilo2025_match_coverage.html")

## 2. Compare the score inputs

The Frey–Osborne probability and ILO task-exposure mean come from different methods and target different concepts. This plot compares their occupation rankings; it does not validate either measure.

In [ ]:
matched = model_inputs[model_inputs["ilo_2025_task_mean"].notna()].copy()
rho = matched[["frey_osborne_probability", "ilo_2025_task_mean"]].corr(method="spearman").iloc[0, 1]
print(f"Matched sample n={len(matched)}; Spearman rank correlation={rho:.3f}")
print(f"Mean Frey–Osborne={matched['frey_osborne_probability'].mean():.3f}; mean ILO={matched['ilo_2025_task_mean'].mean():.3f}")

fig = go.Figure()
labels = {"unique_code_crosswalk": "Unique code match", "ambiguous_code_crosswalk_mean": "Multiple candidates averaged"}
for status, group in matched.groupby("match_status"):
    custom = group[["project_occupation", "isco08_candidates", "isco08_candidate_titles", "candidate_count", "ilo_2025_task_mean"]].to_numpy()
    fig.add_trace(go.Scatter(
        x=group["frey_osborne_probability"], y=group["ilo_2025_task_mean"],
        mode="markers", name=labels.get(status, status), customdata=custom,
        marker=dict(size=9, opacity=.72),
        hovertemplate=("<b>%{customdata[0]}</b><br>Frey–Osborne: %{x:.1%}"+
                       "<br>ILO task mean: %{y:.3f}<br>Match candidates: %{customdata[3]}"+
                       "<br>ISCO-08: %{customdata[1]}<br>%{customdata[2]}<extra></extra>")))
fig.update_layout(title="Frey–Osborne vs ILO 2025 occupation scores", template="plotly_white",
                  xaxis_title="Frey–Osborne probability", yaxis_title="ILO 2025 task-exposure mean",
                  hovermode="closest", dragmode="zoom", height=650)
show_interactive(fig, "ilo2025_vs_frey_osborne_scores.html")

### Ridge-model estimates for the two unmatched occupations

In [ ]:
model_filled = model_inputs[model_inputs["ilo_2025_model_estimate"].notna()][
    ["project_occupation", "ilo_2025_model_estimate", "score_input_method"]
]
display(model_filled)

## 3. Run a demand shock: aggregate scenarios

In [ ]:
def agg_rate(df, start, end):
    e = df.loc[start:end-1, "employment_total"].sum()
    u = df.loc[start:end-1, "unemployment_total"].sum()
    return 100 * u / (e + u)

base_fo = agg_rate(ts_fo, 25, 75)
base_ilo = agg_rate(ts_ilo, 25, 75)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=.12,
                    subplot_titles=["(a) Aggregate unemployment rate", "(b) Change from baseline"])
for df, name, color, baseline in [
    (ts_fo, "Frey–Osborne", "#c45b4d", base_fo),
    (ts_ilo, "ILO 2025", "#4776a8", base_ilo),
]:
    fig.add_trace(go.Scatter(x=df["time_step"], y=df["unemployment_rate_pct"],
        mode="lines", name=name, line=dict(color=color, width=2),
        customdata=np.column_stack([df["employment_total"], df["unemployment_total"]]),
        hovertemplate="Step %{x}<br>Unemployment: %{y:.3f}%<br>Employment: %{customdata[0]:,.0f}"+
                      "<br>Unemployed: %{customdata[1]:,.0f}<extra>%{fullData.name}</extra>"), row=1, col=1)
    fig.add_trace(go.Scatter(x=df["time_step"], y=df["unemployment_rate_pct"]-baseline,
        mode="lines", name=name, legendgroup=name, showlegend=False,
        line=dict(color=color, width=2),
        hovertemplate="Step %{x}<br>Change from baseline: %{y:+.3f} percentage points<extra>%{fullData.name}</extra>"), row=2, col=1)
for row in (1, 2):
    fig.add_vline(x=100, line_dash="dash", line_color="gray", line_width=1, row=row, col=1)
fig.add_hline(y=0, line_dash="dot", line_color="gray", line_width=1, row=2, col=1)
fig.update_yaxes(title_text="Unemployment (%)", row=1, col=1)
fig.update_yaxes(title_text="Change (percentage points)", row=2, col=1)
fig.update_xaxes(title_text="Model time step", row=2, col=1)
fig.update_layout(title="Labor-model scenarios: hover, zoom, and pan", template="plotly_white",
                  height=820, hovermode="x unified", dragmode="zoom",
                  legend=dict(orientation="h", y=1.08, x=0))
show_interactive(fig, "ilo2025_unemployment_scenarios.html")

comparison = pd.DataFrame({
    "scenario": ["Frey–Osborne", "ILO 2025"],
    "baseline unemployment (%)": [base_fo, base_ilo],
    "transition unemployment (%)": [agg_rate(ts_fo, 100, 331), agg_rate(ts_ilo, 100, 331)],
})
comparison["change (percentage points)"] = comparison["transition unemployment (%)"] - comparison["baseline unemployment (%)"]
display(comparison.round(3))

### Why the ILO shock is smaller: interactive score and demand charts

The model preserves total labor demand and redistributes it across occupations. The interactive charts compare exposure-score distributions and occupation-level target-demand shifts. Hover for occupation details; use the legend to toggle series, scroll to zoom, and drag to pan or select. Standalone HTML charts are saved under `extensions/onet_task_exposure/results/interactive/`.


In [ ]:
# Score distributions and the model's normalized occupation-demand shifts.
employment_2016 = pd.read_csv(ROOT / "data" / "ipums_employment_2016.csv")[
    "IPUMS_CPS_av_monthly_employment_whole_period"
].to_numpy(dtype=float)
demand_initial = employment_2016 * (1 + 0.01600001)
def normalized_target_demand(score):
    reduced = demand_initial * (1 - np.asarray(score, dtype=float))
    return reduced * demand_initial.sum() / reduced.sum()

fo_score = model_inputs["frey_osborne_probability"].to_numpy(float)
ilo_score = model_inputs["ilo_2025_score_for_model"].to_numpy(float)
fo_target = normalized_target_demand(fo_score)
ilo_target = normalized_target_demand(ilo_score)
fo_shift_pct = 100 * (fo_target - demand_initial) / demand_initial
ilo_shift_pct = 100 * (ilo_target - demand_initial) / demand_initial

fig = make_subplots(rows=1, cols=2, subplot_titles=["(a) Input score distributions", "(b) Final target-demand shifts"])
for values, name, color in [(fo_score,"Frey–Osborne","#c45b4d"),(ilo_score,"ILO 2025","#4776a8")]:
    fig.add_trace(go.Histogram(x=values, name=name, opacity=.65, histnorm="probability density",
        marker_color=color, nbinsx=30, hovertemplate="Score range: %{x}<br>Density: %{y:.3f}<extra>%{fullData.name}</extra>"), row=1,col=1)
for values, name, color in [(fo_shift_pct,"Frey–Osborne","#c45b4d"),(ilo_shift_pct,"ILO 2025","#4776a8")]:
    fig.add_trace(go.Histogram(x=values, name=name, opacity=.65, histnorm="probability density",
        marker_color=color, nbinsx=30, showlegend=False,
        hovertemplate="Demand shift: %{x:.1f}%<br>Density: %{y:.3f}<extra>%{fullData.name}</extra>"), row=1,col=2)
fig.add_vline(x=0, line_color="gray", line_dash="dot", row=1, col=2)
fig.update_xaxes(title_text="Exposure score (0–1)", row=1,col=1)
fig.update_xaxes(title_text="Change in target demand from initial (%)", row=1,col=2)
fig.update_yaxes(title_text="Density", row=1,col=1)
fig.update_yaxes(title_text="Density", row=1,col=2)
fig.update_layout(title="Score spread and resulting demand reallocation", barmode="overlay",
                  template="plotly_white", height=520, hovermode="closest", dragmode="zoom",
                  legend=dict(orientation="h", y=1.12, x=0))
show_interactive(fig, "ilo2025_score_and_demand_distributions.html")

custom = np.column_stack([model_inputs["project_occupation"], fo_score, ilo_score,
                          model_inputs["score_input_method"]])
limit = max(np.max(np.abs(fo_shift_pct)), np.max(np.abs(ilo_shift_pct))) * 1.05
fig = go.Figure()
fig.add_trace(go.Scatter(x=fo_shift_pct, y=ilo_shift_pct, mode="markers", customdata=custom,
    marker=dict(size=9, color=ilo_score, colorscale="Blues", colorbar=dict(title="ILO score"), opacity=.75),
    hovertemplate="<b>%{customdata[0]}</b><br>FO score: %{customdata[1]:.3f}"+
                  "<br>ILO score: %{customdata[2]:.3f}<br>Score source: %{customdata[3]}"+
                  "<br>FO demand shift: %{x:.1f}%<br>ILO demand shift: %{y:.1f}%<extra></extra>"))
fig.add_trace(go.Scatter(x=[-limit,limit], y=[-limit,limit], mode="lines", name="Equal shift",
                         line=dict(color="gray",dash="dash"), hoverinfo="skip"))
fig.update_layout(title="Occupation-level target-demand shifts: FO vs ILO", template="plotly_white",
    xaxis_title="Frey–Osborne target-demand change (%)", yaxis_title="ILO target-demand change (%)",
    height=650, hovermode="closest", dragmode="zoom")
show_interactive(fig, "ilo2025_occupation_demand_shift_comparison.html")
for label, target in (("Frey–Osborne", fo_target), ("ILO 2025", ilo_target)):
    shift_share = np.abs(target-demand_initial).sum()/demand_initial.sum()
    print(f"{label}: total absolute demand reallocation = {shift_share:.1%} of initial demand")

## 4. Beveridge curve: article replication and automation scenarios

The article's Figure 3 curve is generated by its calibrated sinusoidal aggregate-demand cycle. It is independent of the Frey–Osborne and ILO exposure scores. The first chart reproduces that model cycle using the authors' saved OMN simulation and their 2007–2018 U.S. comparison series. The second chart shows our FO and ILO one-time automation-shock trajectories separately; those are extensions, not the Figure 3 calibration.

In [ ]:
# Article Figure 3c: use the authors’ saved sinusoidal-cycle simulation.
# These files are under the original results/csv directory and are read-only inputs.
import re
from pathlib import Path

article_params = "_deltau160v120gamma160_tau3_amplitude065_period103_dimfact0.csv"
article_u = pd.read_csv(ROOT / "results" / "csv" / ("u_per_occ_numOMNbeveridgeCurve" + article_params))
article_e = pd.read_csv(ROOT / "results" / "csv" / ("e_per_occ_numOMNbeveridgeCurve" + article_params))
article_v = pd.read_csv(ROOT / "results" / "csv" / ("v_per_occ_numOMNbeveridgeCurve" + article_params))

def numeric_model_matrix(frame):
    return frame.iloc[:, 3:].to_numpy(dtype=float)

u_all = numeric_model_matrix(article_u).sum(axis=0)
e_all = numeric_model_matrix(article_e).sum(axis=0)
v_all = numeric_model_matrix(article_v).sum(axis=0)
initial_labor_force = u_all[0] + e_all[0]
article_u_pct = 100 * u_all / initial_labor_force
article_v_pct = 100 * v_all / (v_all + e_all)
# The authors plot one modeled cycle beginning at step 210; period=103 steps.
cycle_start, cycle_period = 210, 103
cycle_steps = np.arange(cycle_start, cycle_start + cycle_period)
article_cycle = pd.DataFrame({
    "time_step": cycle_steps,
    "unemployment_rate_pct": article_u_pct[cycle_steps],
    "vacancy_rate_pct": article_v_pct[cycle_steps],
})
article_cycle.to_csv(RESULTS / "article_figure3c_beveridge_cycle.csv", index=False)

# Reconstruct the article's historical monthly series and retain the plotted 2007–2018 span.
u_source = pd.read_csv(ROOT / "data" / "Total_unemployment_BLS1950-2018.csv")
u_source = u_source[u_source.Year >= 2001]
history_u = []
for _, row in u_source.iterrows(): history_u.extend(row.iloc[1:].to_list())
v_source = pd.read_csv(ROOT / "data" / "vacancy_rateDec2000.csv").iloc[1:]
history_v = v_source["JobOpenings"].to_list()
history_u, history_v = history_u[:-3], history_v[:-2]
assert len(history_u) == len(history_v)
history_dates = pd.date_range("2001-01-01", periods=len(history_u), freq="MS")
history = pd.DataFrame({"date": history_dates, "unemployment_rate_pct": history_u, "vacancy_rate_pct": history_v})
history = history[history.date >= "2007-12-01"]
history.to_csv(RESULTS / "article_figure3c_historical_beveridge_curve.csv", index=False)

fig = go.Figure()
fig.add_trace(go.Scatter(x=history.unemployment_rate_pct, y=history.vacancy_rate_pct,
    mode="lines", name="Historical U.S. (Dec 2007–Sep 2018)", line=dict(color="#888", width=2),
    customdata=history.date.dt.strftime("%Y-%m"),
    hovertemplate="%{customdata}<br>Unemployment %{x:.2f}%<br>Vacancy %{y:.2f}%<extra></extra>"))
fig.add_trace(go.Scatter(x=article_cycle.unemployment_rate_pct, y=article_cycle.vacancy_rate_pct,
    mode="lines+markers", name="Article model: sinusoidal demand cycle (OMN)",
    line=dict(color="#25855a", width=3), marker=dict(size=4), customdata=article_cycle.time_step,
    hovertemplate="Model step %{customdata}<br>Unemployment %{x:.2f}%<br>Vacancy %{y:.2f}%<extra></extra>"))
for i in range(0, len(article_cycle)-1, 10):
    fig.add_annotation(x=article_cycle.unemployment_rate_pct.iloc[i+1], y=article_cycle.vacancy_rate_pct.iloc[i+1],
        ax=article_cycle.unemployment_rate_pct.iloc[i], ay=article_cycle.vacancy_rate_pct.iloc[i],
        xref="x", yref="y", axref="x", ayref="y", showarrow=True, arrowhead=2,
        arrowsize=1, arrowwidth=1.4, arrowcolor="#25855a")
fig.update_layout(title="Beveridge curve: article Figure 3c method", template="plotly_white",
    xaxis_title="Unemployment rate (%)", yaxis_title="Vacancy rate (%)", height=650,
    hovermode="closest", dragmode="zoom")
show_interactive(fig, "article_figure3c_beveridge_curve.html")

# Automation shock paths are a separate extension: they do not reproduce Figure 3c.
path_specs=[
 ("Frey–Osborne", "occupational", "timeseries_frey_osborne_occupational.csv", "#c45b4d", "solid"),
 ("ILO 2025", "occupational", "timeseries_ilo_2025_occupational.csv", "#4776a8", "solid"),
 ("Frey–Osborne", "complete", "timeseries_frey_osborne_complete.csv", "#c45b4d", "dash"),
 ("ILO 2025", "complete", "timeseries_ilo_2025_complete.csv", "#4776a8", "dash"),
]
bev_frames=[]
for scenario, network, filename, color, dash in path_specs:
    frame=pd.read_csv(RESULTS/filename)
    frame["scenario"], frame["network"] = scenario, network
    bev_frames.append(frame)
beveridge_paths=pd.concat(bev_frames, ignore_index=True)
beveridge_paths.to_csv(RESULTS/"ilo2025_frey_osborne_beveridge_paths.csv", index=False)
fig=make_subplots(rows=1, cols=2, shared_xaxes=True, shared_yaxes=True,
    subplot_titles=["(a) Occupational mobility network", "(b) Complete network"])
for col, network in [(1,"occupational"),(2,"complete")]:
    for scenario,color in [("Frey–Osborne","#c45b4d"),("ILO 2025","#4776a8")]:
        g=beveridge_paths[(beveridge_paths.network==network)&(beveridge_paths.scenario==scenario)].iloc[50:]
        fig.add_trace(go.Scatter(x=g.unemployment_rate_pct,y=g.vacancy_rate_pct,mode="lines",
            name=scenario,legendgroup=scenario,showlegend=(col==1),
            line=dict(color=color,width=2,dash="solid" if network=="occupational" else "dash"),
            customdata=g.time_step,hovertemplate="Step %{customdata}<br>Unemployment %{x:.2f}%<br>Vacancy %{y:.2f}%<extra>%{fullData.name}</extra>"),row=1,col=col)
    fig.update_xaxes(title_text="Unemployment rate (%)", row=1,col=col)
    fig.update_yaxes(title_text="Vacancy rate (%)", row=1,col=col)
fig.update_layout(title="Separate automation-shock trajectories in Beveridge space",template="plotly_white",
    height=650,hovermode="closest",dragmode="zoom",legend=dict(orientation="h",y=1.12,x=0))
show_interactive(fig,"ilo2025_frey_osborne_beveridge_shock_paths.html")

summary=[]
for scenario,network,filename,color,dash in path_specs:
    g=pd.read_csv(RESULTS/filename)
    base=g.loc[25:74,"unemployment_rate_pct"].mean()
    shock=g[g.time_step>=100]
    idx=shock.unemployment_rate_pct.idxmax()
    summary.append({"scenario":scenario,"network":network,"baseline unemployment (%)":base,
        "peak unemployment (%)":g.loc[idx,"unemployment_rate_pct"],
        "peak increase (percentage points)":g.loc[idx,"unemployment_rate_pct"]-base,
        "vacancy rate at unemployment peak (%)":g.loc[idx,"vacancy_rate_pct"],"peak step":int(g.loc[idx,"time_step"])})
beveridge_summary=pd.DataFrame(summary)
display(beveridge_summary.round(3))


## 5. Recreate Figure 1(b): separate occupation mobility maps

The same occupational mobility network is shown as two independent interactive maps: one colored by ILO 2025 task exposure and one by Frey–Osborne probability. Each map has its own scale and standalone HTML file. Hover for both scores and ILO crosswalk details; drag to pan and scroll to zoom.

In [ ]:
import networkx as nx
from plotly.colors import sample_colorscale
weights=np.loadtxt(ROOT/"data"/"occupational_mobility_network.csv",delimiter=",")
meta=pd.read_csv(ROOT/"data"/"ipums_variables.csv").sort_values("id").reset_index(drop=True)
occ=model_inputs.sort_values("project_occ_id").reset_index(drop=True)
assert weights.shape==(len(meta),len(meta))==(len(occ),len(occ))
W=weights.copy(); np.fill_diagonal(W,0)
G=nx.from_numpy_array((W+W.T)/2); pos=nx.spring_layout(G,seed=42,iterations=250,weight="weight")
xy=np.array([pos[i] for i in range(len(meta))]); src,dst=np.nonzero(W>0)
p_ilo=occ.ilo_2025_score_for_model.to_numpy(float); p_fo=occ.frey_osborne_probability.to_numpy(float)
custom=np.column_stack([meta.label,meta.total_employment_2010,p_ilo,p_fo,occ.score_input_method,occ.isco08_candidates])
node_size=2*np.sqrt(2.5*np.log1p(meta.total_employment_2010.to_numpy(float)))
for score,title,cbtitle,filename in [
 (p_ilo,"Occupational mobility network · ILO 2025 task exposure","ILO 2025<br>task exposure","ilo2025_occupation_network.html"),
 (p_fo,"Occupational mobility network · Frey–Osborne probability","Frey–Osborne<br>probability","frey_osborne_occupation_network.html")]:
 fig=go.Figure(); bands=np.minimum(((score[src]+score[dst])/2*20).astype(int),19)
 for b in range(20):
  keep=bands==b
  if not keep.any(): continue
  ex=np.column_stack([xy[src[keep],0],xy[dst[keep],0],np.full(keep.sum(),np.nan)]).ravel()
  ey=np.column_stack([xy[src[keep],1],xy[dst[keep],1],np.full(keep.sum(),np.nan)]).ravel()
  fig.add_trace(go.Scattergl(x=ex,y=ey,mode="lines",line=dict(color=sample_colorscale("RdYlBu_r",[(b+.5)/20])[0],width=.45),opacity=.14,hoverinfo="skip",showlegend=False))
 map_custom=np.column_stack([custom,score])
 fig.add_trace(go.Scattergl(x=xy[:,0],y=xy[:,1],mode="markers",name="Occupations",customdata=map_custom,
  marker=dict(size=node_size,color=score,cmin=0,cmax=1,colorscale="RdYlBu_r",colorbar=dict(title=cbtitle,tickformat=".0%"),line=dict(color="white",width=.25),opacity=.95),
  hovertemplate="<b>%{customdata[0]}</b><br>Map score: %{customdata[6]:.1%}<br>ILO score: %{customdata[2]:.1%}<br>Frey–Osborne: %{customdata[3]:.1%}"+
   "<br>Employment: %{customdata[1]:,.0f}<br>ILO match: %{customdata[4]}<br>ISCO candidates: %{customdata[5]}<extra></extra>"))
 fig.update_layout(title=title,template="plotly_white",height=950,hovermode="closest",dragmode="pan",
  xaxis=dict(visible=False,scaleanchor="y",scaleratio=1),yaxis=dict(visible=False),margin=dict(t=75))
 show_interactive(fig,filename)
print(f"{len(meta)} occupations; {len(src):,} directed mobility links; self-links hidden.")

## 6. The 30 highest and 30 lowest ILO exposure scores

Rank all 464 occupation groups. Red marks the two model-filled estimates; blue marks code-crosswalk scores. Hover for candidate ISCO codes and match method.

In [ ]:
ranked=model_inputs.copy()
ranked=ranked.sort_values(["ilo_2025_score_for_model","project_occupation"],ascending=[False,True]).reset_index(drop=True)
high=ranked.head(30); low=ranked.tail(30).sort_values("ilo_2025_score_for_model")
fig=make_subplots(rows=1,cols=2,subplot_titles=["Highest 30 ILO exposure scores","Lowest 30 ILO exposure scores"],horizontal_spacing=.22)
for frame,col in [(high,1),(low,2)]:
 colors=["#cf4446" if "model_estimate" in m else "#2878ad" for m in frame.score_input_method]
 custom=frame[["score_input_method","isco08_candidates","candidate_count"]].to_numpy()
 fig.add_trace(go.Bar(x=frame.ilo_2025_score_for_model,y=frame.project_occupation,orientation="h",marker_color=colors,
  customdata=custom,showlegend=False,hovertemplate="<b>%{y}</b><br>ILO score: %{x:.3f}<br>Method: %{customdata[0]}<br>ISCO candidates: %{customdata[1]} (%{customdata[2]})<extra></extra>"),row=1,col=col)
 fig.update_yaxes(autorange="reversed",tickfont=dict(size=9),row=1,col=col)
for c in (1,2): fig.update_xaxes(title_text="ILO task-exposure mean",range=[0,1],row=1,col=c)
fig.update_layout(title="Occupation rankings by ILO 2025 task exposure",template="plotly_white",height=1100,hovermode="closest",dragmode="zoom",margin=dict(l=260,r=40,t=100))
show_interactive(fig,"ilo2025_highest_lowest_30.html")

## 7. ILO and Frey–Osborne shock: occupation-level unemployment effects

This recreates the original occupation-level comparison for both exposure inputs. Rows show unemployment and long-term unemployment; columns compare Frey–Osborne and ILO 2025. Occupational-network bubbles are sized by employment and colored by median earnings; complete-network points provide the comparison. Hover for occupation details.

In [ ]:
from plotly.subplots import make_subplots
legacy=pd.read_csv(ROOT/"data"/"ipums_variables.csv")
emp=pd.read_csv(ROOT/"data"/"ipums_employment_2016.csv")
assert np.array_equal(legacy.acs_occ_code,emp.ipums_code)
economic=legacy[["id","label","median_earnings"]].copy()
economic["employment"]=emp["IPUMS_CPS_av_monthly_employment_whole_period"].to_numpy(float)
plot_df=outcomes.merge(economic,left_on="project_occ_id",right_on="id",validate="many_to_one")
plot_df=plot_df.merge(model_inputs[["project_occ_id","frey_osborne_probability","ilo_2025_score_for_model","score_input_method"]],on="project_occ_id",validate="many_to_one")
plot_df["exposure_ilo"]=plot_df.ilo_2025_score_for_model
log_emp=np.log1p(plot_df.employment.to_numpy()); sizes=12+72*(log_emp-log_emp.min())/np.ptp(log_emp)
fig=make_subplots(rows=2,cols=2,shared_xaxes=True,vertical_spacing=.13,horizontal_spacing=.11,
 subplot_titles=["(a) Unemployment · FO","(b) Unemployment · ILO 2025","(c) Long-term unemployment · FO","(d) Long-term unemployment · ILO 2025"])
for scenario,col,score in [("Frey-Osborne",1,"frey_osborne_probability"),("ILO 2025 matched + model-filled",2,"exposure_ilo")]:
 part=plot_df[plot_df.scenario==scenario]
 for metric,row in [("unemployment_rate_change_relative_pct",1),("ltu_rate_change_relative_pct",2)]:
  for network,nlabel,color in [("occupational","Occupational mobility network","#218c57"),("complete","Complete network","#d64045")]:
   g=part[part.network==network]; custom=g[["label","employment","median_earnings",score,"score_input_method"]].to_numpy()
   marker=dict(color=color,size=6,opacity=.82)
   if network=="occupational":
    se=12+72*(np.log1p(g.employment)-log_emp.min())/np.ptp(log_emp)
    marker=dict(size=se,sizemode="area",sizeref=2*sizes.max()/19**2,sizemin=3,
                color=g.median_earnings,coloraxis="coloraxis",opacity=.8)
   fig.add_trace(go.Scatter(x=g[score],y=g[metric],mode="markers",name=nlabel,legendgroup=network,
    showlegend=(row==1 and col==1),marker=marker,customdata=custom,
    hovertemplate="<b>%{customdata[0]}</b><br>Exposure: %{x:.1%}<br>Outcome change: %{y:+.2f}%"+
      "<br>Employment: %{customdata[1]:,.0f}<br>Median earnings: $%{customdata[2]:,.0f}"+
      "<br>ILO score source: %{customdata[4]}<extra>"+nlabel+"</extra>"),row=row,col=col)
  fig.add_hline(y=0,line_dash="dash",line_color="gray",row=row,col=col)
  fig.update_yaxes(title_text="Rate change (%)",row=row,col=col)
for c in (1,2): fig.update_xaxes(title_text="Exposure input",tickformat=".0%",range=[-.03,1.03],row=2,col=c)
fig.update_layout(title="Occupation-level labor-model effects by exposure input",template="plotly_white",height=950,
 hovermode="closest",dragmode="zoom",coloraxis=dict(colorscale="Viridis",cmin=plot_df.median_earnings.min(),
 cmax=plot_df.median_earnings.max(),colorbar=dict(title="Median annual<br>earnings (USD)",tickprefix="$",tickformat=",.0f")),
 legend=dict(orientation="h",y=1.08,x=0),margin=dict(t=140))
show_interactive(fig,"ilo2025_occupation_shock_effects.html")
display(plot_df.groupby(["scenario","network"],as_index=False).agg(
 mean_u_change_pp=("unemployment_change_percentage_points","mean"),mean_ltu_change_pp=("ltu_change_percentage_points","mean")).round(3))

## 8. Figure 6: network effects versus median earnings

For each exposure input, subtract the complete-network outcome change from the occupational-network outcome change. Positive values indicate a larger modeled change under the occupational network. Compare unemployment and long-term unemployment effects by median earnings.

In [ ]:
effects=[]
for scenario,g in plot_df.groupby("scenario"):
 occ=g[g.network=="occupational"].set_index("project_occ_id"); comp=g[g.network=="complete"].set_index("project_occ_id")
 z=occ[["label","median_earnings","employment","frey_osborne_probability","exposure_ilo"]].copy()
 z["u_effect"]=occ.unemployment_rate_change_relative_pct-comp.unemployment_rate_change_relative_pct
 z["ltu_effect"]=occ.ltu_rate_change_relative_pct-comp.ltu_rate_change_relative_pct; z["scenario"]=scenario
 effects.append(z.reset_index())
effects=pd.concat(effects,ignore_index=True)
fig=make_subplots(rows=2,cols=2,shared_xaxes=True,vertical_spacing=.13,horizontal_spacing=.12,
 subplot_titles=["(a) Unemployment · FO","(b) Unemployment · ILO","(c) Long-term unemployment · FO","(d) Long-term unemployment · ILO"])
for scenario,col,score in [("Frey-Osborne",1,"frey_osborne_probability"),("ILO 2025 matched + model-filled",2,"exposure_ilo")]:
 g=effects[effects.scenario==scenario]
 for metric,row in [("u_effect",1),("ltu_effect",2)]:
  size=12+72*(np.log1p(g.employment)-np.log1p(effects.employment).min())/np.ptp(np.log1p(effects.employment))
  custom=g[["label","median_earnings","employment",score,metric]].to_numpy()
  fig.add_trace(go.Scatter(x=g.median_earnings/1000,y=g[metric],mode="markers",showlegend=False,
   marker=dict(size=size,sizemode="area",sizeref=2*size.max()/19**2,sizemin=3,color=g[score],colorscale="RdYlBu_r",cmin=0,cmax=1,opacity=.76),
   customdata=custom,hovertemplate="<b>%{customdata[0]}</b><br>Earnings: $%{customdata[1]:,.0f}<br>Employment: %{customdata[2]:,.0f}"+
   "<br>Exposure: %{customdata[3]:.1%}<br>Network effect: %{y:+.2f} pp of relative change<extra></extra>"),row=row,col=col)
  fig.add_hline(y=0,line_dash="dash",line_color="gray",row=row,col=col)
  fig.update_yaxes(title_text="Network effect (pp of relative change)",row=row,col=col)
for c in (1,2): fig.update_xaxes(title_text="Median annual earnings (USD thousands)",row=2,col=c)
fig.update_layout(title="Figure 6 · Network effects versus median earnings",template="plotly_white",height=950,hovermode="closest",dragmode="zoom",margin=dict(t=130))
show_interactive(fig,"ilo2025_figure6_network_effects_wages.html")

## 9. Potential automation traps: separate score maps

The potential-trap view combines exposure with limited access to lower-exposure occupations through historical mobility links. Separate interactive maps show ILO and Frey–Osborne exposure; bubble color indicates the long-term-unemployment network effect. This is an exploratory ranking, not a job-loss prediction.

In [ ]:
raw=np.loadtxt(ROOT/"data"/"occupational_mobility_network.csv",delimiter=","); assert raw.shape==(464,464)
A=raw/raw.sum(axis=1,keepdims=True); B=A.copy(); np.fill_diagonal(B,0)
trap=plot_df[plot_df.network=="occupational"].merge(
 effects[["project_occ_id","scenario","ltu_effect"]],on=["project_occ_id","scenario"],validate="one_to_one")
for key,col,model_col in [("ilo","exposure_ilo","ilo_2025_score_for_model"),("fo","frey_osborne_probability","frey_osborne_probability")]:
 exposure_by_id=model_inputs.sort_values("project_occ_id")[model_col].to_numpy(float)
 access=B@(1-exposure_by_id); access_by_id=dict(enumerate(access))
 trap[f"safe_access_{key}"]=trap.project_occ_id.map(access_by_id)
 trap[f"trap_score_{key}"]=trap[col]*(1-trap[f"safe_access_{key}"])
for scenario,key,score,title,filename in [
 ("ILO 2025 matched + model-filled","ilo","exposure_ilo","Potential automation traps · ILO 2025","ilo2025_potential_automation_traps.html"),
 ("Frey-Osborne","fo","frey_osborne_probability","Potential automation traps · Frey–Osborne","frey_osborne_potential_automation_traps.html")]:
 g=trap[trap.scenario==scenario]
 size=12+72*(np.log1p(g.employment)-np.log1p(trap.employment).min())/np.ptp(np.log1p(trap.employment))
 custom=g[["label",score,f"trap_score_{key}",f"safe_access_{key}","ltu_effect","median_earnings","employment","score_input_method"]].to_numpy()
 fig=go.Figure(go.Scatter(x=g[f"safe_access_{key}"],y=g[score],mode="markers",customdata=custom,
  marker=dict(size=size,sizemode="area",sizeref=2*size.max()/19**2,sizemin=3,color=g.ltu_effect,colorscale="RdBu_r",cmid=0,
   colorbar=dict(title="LTU network<br>effect (%)"),opacity=.8),
  hovertemplate="<b>%{customdata[0]}</b><br>Exposure: %{y:.1%}<br>Access to safer work: %{x:.3f}"+
   "<br>Trap score: %{customdata[2]:.3f}<br>LTU network effect: %{customdata[4]:+.2f}%"+
   "<br>Earnings: $%{customdata[5]:,.0f}<br>Employment: %{customdata[6]:,.0f}<br>Score source: %{customdata[7]}<extra></extra>"))
 fig.update_layout(title=title,template="plotly_white",height=750,hovermode="closest",dragmode="zoom",
  xaxis=dict(title="Access to safer occupations (higher is better)",range=[-.03,1.03]),
  yaxis=dict(title="Exposure",tickformat=".0%",range=[-.03,1.03]))
 show_interactive(fig,filename)
trap_rank=(trap[trap.scenario=="ILO 2025 matched + model-filled"].sort_values("trap_score_ilo",ascending=False)
 [["label","exposure_ilo","safe_access_ilo","trap_score_ilo","ltu_effect","score_input_method"]].head(30).reset_index(drop=True))
trap_rank.insert(0,"rank",np.arange(1,len(trap_rank)+1)); display(trap_rank.round(3))

## Reproduce the analysis

From the repository root run:

```powershell
python extensions/onet_task_exposure/match_ilo_to_project.py
python extensions/onet_task_exposure/train_exposure_model.py
python extensions/onet_task_exposure/run_ilo_model_scenario.py
```

Outputs and standalone interactive HTML charts are under `extensions/onet_task_exposure/results/`. The ILO input uses 462 crosswalk means and 2 model estimates. This scenario analysis is not a forecast.